In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms as T

from PIL import Image
from collections import deque
from torchmetrics import Metric
from torchmetrics.classification import MulticlassJaccardIndex


In [ ]:
p1 = "/data/video-segmentation/cityscapes/leftImg8bit_sequence/train/aachen/aachen_000000_000019_leftImg8bit.png"
p2 = "/data/video-segmentation/cityscapes/leftImg8bit_sequence/train/aachen/aachen_000000_000023_leftImg8bit.png"

img1 = Image.open(p1)
img1

In [ ]:
label_path = "/data/video-segmentation/cityscapes/gtFine/train/aachen/aachen_000000_000019_gtFine_labelTrainIds.png"

a = Image.open(label_path)
t = T.PILToTensor()
label = t(a).to(torch.long)

In [ ]:
test = torch.randint(0, 5, (3, 17, 23))

In [ ]:
F.one_hot(test).shape

In [ ]:


class SegIoU(Metric):
    def __init__(self, num_classes=19, ignore_index=255, boundary_ratio=0.02):
        super().__init__()

        self.C = num_classes
        self.ignore_index = ignore_index
        self.boundary_ratio = boundary_ratio

        self.iou = MulticlassJaccardIndex(
            num_classes, ignore_index=ignore_index, average=None, zero_division=float("nan")
        )
        self.add_state("b_inter", torch.zeros(num_classes), dist_reduce_fx="sum")
        self.add_state("b_union", torch.zeros(num_classes), dist_reduce_fx="sum")

    def _band(self, mask, d):
        m = F.pad(mask, (d,) * 4, value=0.0)
        eroded = 1 - F.max_pool2d(1 - m, 2 * d + 1, stride=1)
        return mask.bool() & ~eroded.bool()

    def update(self, pred, target):
        H, W = target.shape[-2:]
        if pred.dim() == 4:
            if pred.shape[-2:] != (H, W):
                pred = F.interpolate(pred, size=(H, W), mode="bilinear", align_corners=False)
            pred = pred.argmax(1)

        self.iou.update(pred, target)

        d = max(1, round(self.boundary_ratio * (H ** 2 + W ** 2) ** 0.5))
        valid = (target != self.ignore_index).unsqueeze(1)
        gt = F.one_hot(target.masked_fill(~valid.squeeze(1), self.C), self.C + 1)[..., : self.C]
        pr = F.one_hot(pred, self.C)
        gb = self._band(gt.permute(0, 3, 1, 2).float(), d)
        pb = self._band(pr.permute(0, 3, 1, 2).float(), d)
        self.b_inter += (gb & pb & valid).sum((0, 2, 3))
        self.b_union += ((gb | pb) & valid).sum((0, 2, 3))

    def compute(self):
        per_class = self.iou.compute()
        biou = self.b_inter / self.b_union  # 0/0 -> nan for absent classes
        return {
            "mIoU": per_class.nanmean(),
            "IoU": per_class,
            "mBIoU": biou.nanmean(),
            "BIoU": biou,
        }

    def reset(self):
        super().reset()
        self.iou.reset()

In [ ]:
label = label.to("cuda")
pred = torch.randint(0, 19, label.shape, dtype=torch.long).to("cuda")


In [ ]:
seg_metric = SegIoU().to("cuda")

seg_metric.update(pred, label)
seg_metric.compute()#.item()


In [ ]:



def _grid(B, H, W, device):
    ys, xs = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device), indexing="ij")
    return torch.stack((xs, ys), 0).float().unsqueeze(0).expand(B, -1, -1, -1)


def warp(x, flow, mode="bilinear"):
    """Sample x at p + flow(p) for each pixel p. x: (B,C,H,W), flow: (B,2,H,W) in pixels."""
    B, _, H, W = flow.shape
    pos = _grid(B, H, W, flow.device) + flow
    grid = torch.stack((2 * pos[:, 0] / (W - 1) - 1, 2 * pos[:, 1] / (H - 1) - 1), dim=-1)
    out = F.grid_sample(x, grid, mode=mode, padding_mode="zeros", align_corners=True)
    return out, pos


def valid_mask(flow_bw, flow_fw, thresh=1.0):
    """flow_bw: t -> t-1, flow_fw: t-1 -> t. True where flow is in-bounds and fwd-bwd consistent."""
    H, W = flow_bw.shape[-2:]
    fw_at, pos = warp(flow_fw, flow_bw, "bilinear")
    err = (flow_bw + fw_at).norm(dim=1)
    inb = (pos[:, 0] >= 0) & (pos[:, 0] <= W - 1) & (pos[:, 1] >= 0) & (pos[:, 1] <= H - 1)
    return (err < thresh) & inb


class WarpedTC(Metric):
    is_differentiable = False
    higher_is_better = True
    full_state_update = False

    def __init__(self, num_classes=19, fb_thresh=1.0, **kwargs):
        super().__init__(**kwargs)
        self.C = num_classes
        self.fb_thresh = fb_thresh
        self.add_state("conf", torch.zeros(num_classes, num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("valid_px", torch.tensor(0, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("total_px", torch.tensor(0, dtype=torch.long), dist_reduce_fx="sum")

    def update(self, pred_t, pred_prev, flow_bw, flow_fw):
        """pred_t, pred_prev: (B,H,W) class ids. flow_bw: t->t-1, flow_fw: t-1->t, (B,2,H,W)."""
        warped, _ = warp(pred_prev.unsqueeze(1).float(), flow_bw, "nearest")
        warped = warped.squeeze(1).long()
        valid = valid_mask(flow_bw, flow_fw, self.fb_thresh)

        idx = pred_t[valid] * self.C + warped[valid]
        self.conf += torch.bincount(idx, minlength=self.C ** 2).reshape(self.C, self.C)
        self.valid_px += valid.sum()
        self.total_px += valid.numel()

    def compute(self):
        conf = self.conf.double()
        tp = conf.diag()
        iou = tp / (conf.sum(0) + conf.sum(1) - tp)  # 0/0 -> nan for absent classes
        return {"TC": iou.nanmean(), "TC_per_class": iou,
                "coverage": self.valid_px.double() / self.total_px}

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

weights = Raft_Large_Weights.DEFAULT
raft = raft_large(weights=weights).eval().cuda()
raft_tf = weights.transforms()

@torch.inference_mode()
def compute_flow(src, dst):
    """Flow src -> dst. src, dst: (B,3,H,W) float in [0,1]; H, W divisible by 8."""
    a, b = raft_tf(src, dst)
    return raft(a, b)[-1]

def resize_flow(flow, size):
    """Upsample flow to size (H,W), scaling vectors to match."""
    h, w = flow.shape[-2:]
    out = F.interpolate(flow, size=size, mode="bilinear", align_corners=False)
    out[:, 0] *= size[1] / w
    out[:, 1] *= size[0] / h
    return out

In [ ]:
tc = WarpedTC().cuda()

@torch.inference_mode()
def eval_snippet(frames):  # frames: (T,3,H,W) float in [0,1] on cuda
    preds = torch.cat([to_pred(model(f[None]), f.shape[-2:]) for f in frames])
    for t in range(1, len(frames)):
        prev, cur = frames[t - 1 : t], frames[t : t + 1]
        flow_bw = compute_flow(cur, prev)   # t -> t-1
        flow_fw = compute_flow(prev, cur)   # t-1 -> t
        tc.update(preds[t : t + 1], preds[t - 1 : t], flow_bw, flow_fw)

# for frames in snippets: eval_snippet(frames)
# print(tc.compute())

In [ ]:
torch.manual_seed(0)
B, H, W = 1, 128, 256
p = F.interpolate(torch.randint(0, 19, (B, 1, 8, 16)).float(), size=(H, W)).squeeze(1).long()
prev = torch.roll(p, 4, dims=-1)                      # objects were 4 px to the right at t-1
flow_bw = torch.zeros(B, 2, H, W); flow_bw[:, 0] = 4  # t -> t-1: +4 in x
flow_fw = -flow_bw

m = WarpedTC(); m.update(p, prev, flow_bw, flow_fw)
print("correct flow:", m.compute())                   # TC = 1.0, coverage < 1 (right edge masked)

m = WarpedTC(); z = torch.zeros_like(flow_bw); m.update(p, prev, z, z)
print("zero flow:   ", m.compute())                   # TC < 1

In [ ]:

class VEC(Metric):
    """Video Effective Consistency (DTERN, ICCV 2025), sliding window, dataset-level accumulation.

    Call update(pred, target) once per frame, in order. Call new_sequence() before each new video.
    """
    is_differentiable = False
    higher_is_better = True
    full_state_update = False

    def __init__(self, num_classes=19, window=8, ignore_index=255, **kwargs):
        super().__init__(**kwargs)
        self.C, self.n, self.ignore_index = num_classes, window, ignore_index
        self.add_state("inter", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("gt_cnt", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("pr_cnt", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self._buf = deque(maxlen=window)

    def new_sequence(self):
        self._buf.clear()

    def update(self, pred, target):
        """pred, target: (H,W) class ids for one frame."""
        self._buf.append((pred, target))
        if len(self._buf) < self.n:
            return

        preds = torch.stack([p for p, _ in self._buf])    # (n,H,W)
        gts = torch.stack([g for _, g in self._buf])
        g0, p0 = gts[0], preds[0]

        valid = (gts != self.ignore_index).all(0)
        gt_stable = (gts == g0).all(0) & valid
        pr_stable = (preds == p0).all(0) & valid
        both = gt_stable & pr_stable & (g0 == p0)

        C = self.C
        self.inter += torch.bincount(g0[both], minlength=C)[:C]
        self.gt_cnt += torch.bincount(g0[gt_stable], minlength=C)[:C]
        self.pr_cnt += torch.bincount(p0[pr_stable], minlength=C)[:C]

    def compute(self):
        inter = self.inter.double()
        union = self.gt_cnt.double() + self.pr_cnt.double() - inter
        iou = inter / union  # 0/0 -> nan for absent classes
        return {"VEC": iou.nanmean(), "VEC_per_class": iou}

    def reset(self):
        super().reset()
        self._buf.clear()